# 1. Preprocess: FlowCam exports → particle images

A FlowCam run is saved as **mosaic images** (`<run>_000001.tif`, `<run>_000002.tif`, …) plus a
**CSV with one row per particle**, giving the mosaic it sits on (`Image File`) and its box
(`Image X/Y/Width/Height`). When the particles have been classified by hand, the CSV also has a
`Class` column.

This notebook cuts every particle out of its mosaic into its own `.tif`, sorted into one folder per
class. That is the layout used for training (`train.ipynb`) and for checking accuracy
(`inference.ipynb`).

**Example data:** the Lake Erie test set, `Erie_test_data/`, with mosaics in `Classified_data/` and
CSVs in `CSV_Data/`.

Requires `pip install hierroute jupyter`. The example data lives at `/data/Hier-Zooplankton` on the lab server; change `DATA_DIR` below if your copy is elsewhere. Everything this notebook writes goes under `WORK_DIR`.

In [ ]:
from pathlib import Path

DATA_DIR = Path("/data/Hier-Zooplankton")        # example dataset (internal)
WORK_DIR = Path.home() / "hierroute_example"      # everything the example notebooks write goes here
WORK_DIR.mkdir(parents=True, exist_ok=True)

EXPORT_DIR = DATA_DIR / "Erie_test_data" / "Classified_data"   # mosaics, one folder per run
CSV_DIR = DATA_DIR / "Erie_test_data" / "CSV_Data"             # the export CSVs
CROPS_DIR = WORK_DIR / "erie_crops"                            # output: one folder per class

## Look at the raw export

Each run folder holds that run's mosaics. `read_flowcam_csv` reads an export CSV and repairs the
quirks FlowCam sometimes writes, such as unquoted commas in run names.

In [ ]:
import pandas as pd
from hierroute.flowcam import read_flowcam_csv

runs = sorted(p for p in EXPORT_DIR.iterdir() if p.is_dir())
print(f"{len(runs)} runs, e.g. {runs[0].name}: {len(list(runs[0].glob('*.tif')))} mosaics")

csv_path = sorted(CSV_DIR.glob("*.csv"))[0]
df = read_flowcam_csv(csv_path)
print(f"{csv_path.name}: {len(df)} particles")
df[["Particle ID", "Class", "Image File", "Image X", "Image Y", "Image Width", "Image Height"]].head()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

# One mosaic with the CSV's particle boxes drawn on it
mosaic_name = df["Image File"].iloc[0]
mosaic_path = next(EXPORT_DIR.rglob(mosaic_name))
rows = df[df["Image File"] == mosaic_name]

fig, ax = plt.subplots(figsize=(12, 7.5))
ax.imshow(Image.open(mosaic_path), cmap="gray")
boxes = rows[["Image X", "Image Y", "Image Width", "Image Height", "Class"]].itertuples(index=False)
for x, y, w, h, cls in boxes:
    ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, edgecolor="red", linewidth=1))
    ax.text(x, y - 4, cls, color="red", fontsize=7)
ax.set_title(f"{mosaic_name}: {len(rows)} particles")
ax.axis("off");

## Build the particle table

`load_particles` reads every CSV, finds each particle's mosaic through its `Image File` column (so CSV
file names don't matter), and attaches the class.

`ClassRules` controls how the CSV's `Class` becomes a folder name:
- by default, `Nauplius` becomes `Nauplius_Copepod`, using the `Group` column (or `Copepod` if there
  is none)
- `rename={...}` renames classes
- `ignore={...}` skips classes

Every other CSV column is kept as a `csv_*` measurement.

In [ ]:
from hierroute.flowcam import ClassRules, load_particles

rules = ClassRules(ignore={"Duplicate"})
table = load_particles(EXPORT_DIR, csv_dir=CSV_DIR, rules=rules)
print(table.summary())
table.particles["label"].value_counts()

## Write the crops

Each particle is saved as `CROPS_DIR/<class>/<mosaic name>_<Particle ID>.tif`.

`write_crops` returns a manifest: the particle table plus where each crop went, whether it was
written, and whether its box ran past the mosaic edge. Save it as `manifest.csv` next to the crops.
`hierroute evaluate` then picks up the FlowCam measurements from it automatically.

Crops that already exist are skipped, so re-running is cheap (`overwrite=True` to redo them).

In [ ]:
from hierroute.flowcam import write_crops

manifest = write_crops(table, CROPS_DIR, by="label", workers=8)
manifest.to_csv(CROPS_DIR / "manifest.csv", index=False)

print(manifest["crop_status"].value_counts().to_dict())
print(f"{int(manifest['out_of_bounds'].sum())} boxes extended past their mosaic edge (zero-padded)")

In [ ]:
# A few example crops per class
classes = ["Bosmina", "Calanoid", "Cyclopoid", "Daphnia", "Nauplius_Copepod", "Rotifer", "Detritus"]
n = 6
fig, axes = plt.subplots(len(classes), n, figsize=(n * 1.4, len(classes) * 1.5))
for row, cls in zip(axes, classes):
    files = sorted((CROPS_DIR / cls).glob("*.tif"))[:n]
    for ax, f in zip(row, files):
        ax.imshow(Image.open(f).convert("L"), cmap="gray")
    for ax in row:
        ax.set_xticks([]); ax.set_yticks([])
    row[0].set_ylabel(cls, rotation=0, ha="right", va="center", fontsize=9)
plt.tight_layout()

## Unlabelled exports

For exports without manual classes, pass `rules=None` and `by="run"`. The crops then go into one folder
per run. You often don't need this step: `inference.ipynb` classifies raw exports directly, cropping in
memory.

## The same thing from the terminal

```bash
hierroute crop /data/Hier-Zooplankton/Erie_test_data/Classified_data \
    --csv-dir /data/Hier-Zooplankton/Erie_test_data/CSV_Data \
    --ignore-classes Duplicate --out ~/hierroute_example/erie_crops

hierroute crop path/to/export --unlabelled --out crops_by_run/
```

The class folders at the top of `/data/Hier-Zooplankton` (the training data used in `train.ipynb`)
have the same layout.